# L15 - Errore e discesa del gradiente

Eseguire le celle in ordine con `Maiusc + Invio`.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Dati e modello

Ore di studio e punteggio in una prova (dati sintetici). Il modello è una retta, $\hat{y} = w x + b$: il problema è trovare $w$ e $b$ che la avvicinano il più possibile ai punti.

In [ ]:
rng = np.random.default_rng(1)
ore = rng.uniform(0, 10, size=30)
punteggio = 4 * ore + 30 + rng.normal(0, 5, size=30)

plt.scatter(ore, punteggio)
plt.xlabel("ore di studio")
plt.ylabel("punteggio")
plt.show()

## 2. La funzione di perdita

L'errore quadratico medio (MSE, mean squared error) misura quanto le previsioni sono lontane dai valori veri:

$$L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

Più è piccolo, migliore è il modello.

In [ ]:
def mse(y, y_prev):
    return ((y_prev - y) ** 2).mean()

for w, b in [(0, 50), (2, 40), (4, 30)]:
    print(f"w = {w}, b = {b}: perdita = {mse(punteggio, w * ore + b):.1f}")

## 3. La pendenza di una curva

La pendenza di una funzione $f$ in un punto $w$ si stima con il rapporto incrementale, calcolato con un $h$ piccolo:

$$\text{pendenza} \approx \frac{f(w + h) - f(w - h)}{2h}$$

Il suo limite per $h$ che tende a 0 è la derivata. Esempio con $f(w) = (w - 3)^2 + 1$, che ha il minimo in $w = 3$.

In [ ]:
def f(w):
    return (w - 3) ** 2 + 1

h = 0.001
for w in [0, 2, 3, 5]:
    print(f"w = {w}: pendenza circa {(f(w + h) - f(w - h)) / (2 * h):.3f}")

La pendenza è negativa a sinistra del minimo, zero nel minimo, positiva a destra. Per scendere verso il minimo bisogna quindi spostarsi nella direzione opposta al segno della pendenza.

## 4. Discesa del gradiente in una variabile

Regola: $w \leftarrow w - \eta \cdot \text{pendenza}(w)$, ripetuta molte volte. Per $f(w) = (w - 3)^2 + 1$ la derivata è $f'(w) = 2(w - 3)$.

In [ ]:
def df(w):
    return 2 * (w - 3)

w = -1.0
eta = 0.1
percorso = [w]
for passo in range(25):
    w = w - eta * df(w)
    percorso.append(w)
print("w finale:", round(w, 4))

xs = np.linspace(-2, 8, 200)
plt.plot(xs, f(xs), label="f(w)")
plt.plot(percorso, [f(p) for p in percorso], "o-", markersize=4, label="passi della discesa")
plt.xlabel("w")
plt.legend()
plt.show()

## 5. Due parametri: la retta

Per la retta $\hat{y} = w x + b$ con perdita MSE le derivate rispetto a $w$ e a $b$ sono:

$$\frac{\partial L}{\partial w} = \frac{2}{n} \sum (\hat{y}_i - y_i)\, x_i \qquad \frac{\partial L}{\partial b} = \frac{2}{n} \sum (\hat{y}_i - y_i)$$

Il simbolo $\partial$ indica la derivata rispetto a una sola variabile, considerando fisse le altre (derivata parziale). Il vettore delle due derivate si chiama gradiente.

In [ ]:
w, b = 0.0, 0.0
eta = 0.01
perdite = []
for passo in range(2000):
    y_prev = w * ore + b
    errore = y_prev - punteggio
    grad_w = 2 * (errore * ore).mean()
    grad_b = 2 * errore.mean()
    w = w - eta * grad_w
    b = b - eta * grad_b
    perdite.append(mse(punteggio, y_prev))
print(f"w = {w:.2f}, b = {b:.2f}, perdita finale = {perdite[-1]:.1f}")

fig, assi = plt.subplots(1, 2, figsize=(10, 3.5))
assi[0].plot(perdite)
assi[0].set_yscale("log")
assi[0].set_xlabel("passo")
assi[0].set_ylabel("perdita (scala logaritmica)")
assi[1].scatter(ore, punteggio)
xs = np.linspace(0, 10, 50)
assi[1].plot(xs, w * xs + b, color="black")
assi[1].set_xlabel("ore di studio")
fig.tight_layout()
plt.show()

`set_yscale("log")` usa una scala logaritmica sull'asse verticale: ogni tacca vale 10 volte la precedente. Rende visibile la diminuzione della perdita anche quando i valori passano da migliaia a decine.

## Esercizi

**Esercizio 1 (base).** Calcolare `perdita_migliore`, la perdita MSE della retta con i parametri `w` e `b` trovati nella sezione 5, e `perdita_zero`, la perdita della retta con $w = 0$, $b = 0$.

In [ ]:
perdita_migliore = mse(punteggio, w * ore + b)
perdita_zero = mse(punteggio, 0 * ore + 0)
print(perdita_migliore, perdita_zero)

In [ ]:
controlla(1, lambda: np.isclose(perdita_migliore, ((w * ore + b - punteggio) ** 2).mean()) and np.isclose(perdita_zero, (punteggio ** 2).mean()))

**Esercizio 2 (base).** Scrivere `pendenza(g, x, h=0.001)`, che stima la pendenza della funzione `g` nel punto `x` con il rapporto incrementale. Verificarla su $g(x) = x^2$ in $x = 3$ (la derivata esatta vale 6).

In [ ]:
def pendenza(g, x, h=0.001):
    return (g(x + h) - g(x - h)) / (2 * h)

print(pendenza(lambda x: x ** 2, 3))

In [ ]:
controlla(2, lambda: np.isclose(pendenza(lambda x: x ** 2, 3), 6) and np.isclose(pendenza(np.exp, 0), 1))

**Esercizio 3 (standard).** Scrivere `discesa(g, w0, eta, passi)`, che applica la discesa del gradiente alla funzione `g` partendo da `w0`, usando `pendenza` per stimare la derivata, e restituisce la lista di tutti i valori di `w` (compreso quello iniziale). Provarla su `f` partendo da 8.

In [ ]:
def discesa(g, w0, eta, passi):
    w = w0
    valori = [w]
    for _ in range(passi):
        w = w - eta * pendenza(g, w)
        valori.append(w)
    return valori

print(discesa(f, 8, 0.1, 30)[-1])

In [ ]:
controlla(3, lambda: len(discesa(f, 8, 0.1, 30)) == 31 and abs(discesa(f, 8, 0.1, 30)[-1] - 3) < 0.01)

**Esercizio 4 (standard).** Effetto del tasso di apprendimento. Con `discesa` su `f`, partendo da 8 per 20 passi, provare `eta` pari a 0.05, 0.5, 0.95 e 1.05. Salvare l'ultimo valore di `w` nel dizionario `finali` con chiave `eta` e disegnare i quattro percorsi (valore di `w` in funzione del passo) nello stesso grafico. Descrivere in una cella di testo i quattro comportamenti.

In [ ]:
finali = {}
for eta in [0.05, 0.5, 0.95, 1.05]:
    v = discesa(f, 8, eta, 20)
    finali[eta] = v[-1]
    plt.plot(v, marker=".", label=f"eta = {eta}")
plt.axhline(3, color="gray", linestyle="--")
plt.ylim(-10, 16)
plt.xlabel("passo")
plt.ylabel("w")
plt.legend()
plt.show()
print(finali)

In [ ]:
controlla(4, lambda: abs(finali[0.5] - 3) < 1e-3 and abs(finali[0.05] - 3) > 0.5 and abs(finali[1.05] - 3) > 5)

**Esercizio 5 (standard).** Nuovi dati: temperatura in gradi Celsius (`c`) e in Fahrenheit (`fa`), con un piccolo errore di misura. Addestrare la retta `fa = w * c + b` con la discesa del gradiente (come nella sezione 5) e verificare che $w$ si avvicini a 1.8 e $b$ a 32. Usare `eta = 0.001` e 20 000 passi.

In [ ]:
c = np.array([-10, 0, 5, 12, 20, 25, 30, 37])
fa = 1.8 * c + 32 + np.random.default_rng(2).normal(0, 0.5, size=len(c))
w_t, b_t = 0.0, 0.0
for _ in range(20_000):
    err = w_t * c + b_t - fa
    w_t = w_t - 0.001 * 2 * (err * c).mean()
    b_t = b_t - 0.001 * 2 * err.mean()
print(w_t, b_t)

In [ ]:
controlla(5, lambda: abs(w_t - 1.8) < 0.05 and abs(b_t - 32) < 1)

**Esercizio 6 (approfondimento).** Verificare le formule del gradiente della sezione 5: nel punto $w = 1$, $b = 10$ calcolare `grad_w_formula` e `grad_b_formula` con le formule e `grad_w_numerico` e `grad_b_numerico` con il rapporto incrementale della funzione di perdita, variando un parametro alla volta.

In [ ]:
w0, b0 = 1.0, 10.0
e0 = w0 * ore + b0 - punteggio
grad_w_formula = 2 * (e0 * ore).mean()
grad_b_formula = 2 * e0.mean()
grad_w_numerico = pendenza(lambda ww: mse(punteggio, ww * ore + b0), w0)
grad_b_numerico = pendenza(lambda bb: mse(punteggio, w0 * ore + bb), b0)
print(grad_w_formula, grad_w_numerico, grad_b_formula, grad_b_numerico)

In [ ]:
controlla(6, lambda: np.isclose(grad_w_formula, grad_w_numerico, rtol=1e-4) and np.isclose(grad_b_formula, grad_b_numerico, rtol=1e-4) and np.isclose(grad_b_formula, 2 * (1.0 * ore + 10 - punteggio).mean()))